In [125]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re
import csv

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))


from scripts.text_matching import normalise_text

In [126]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
matched_now_file = Path(project_root / "data/people/matched_now.json")
multiple_file = Path(project_root / "data/people/multiples_matched.json")


In [127]:
with open(matched_now_file, "r") as f:
   matched_now = json.load(f)

with open(multiple_file, "r") as f:
   former_multiples = json.load(f)


In [128]:
rprint(dict(list(matched_now.items())[:2]))

{
    '5569': [
        {
            'display_name': 'Reiffenstein, Bruno',
            'composite_id': 'wien_209_9_10',
            'sort_order': 1,
            'is_author': False,
            'is_editor': False,
            'is_contributor': True,
            'is_translator': False
        }
    ],
    '6263': [
        {
            'display_name': 'Schreck, Joachim',
            'composite_id': 'wien_216_9_10',
            'sort_order': 1,
            'is_author': False,
            'is_editor': True,
            'is_contributor': False,
            'is_translator': False
        },
        {
            'display_name': 'Schreck, Joachim',
            'composite_id': 'literatur_254_11_12',
            'sort_order': 1,
            'is_author': False,
            'is_editor': True,
            'is_contributor': False,
            'is_translator': False
        }
    ]
}

In [129]:

for id, entries in former_multiples.items():
    # rprint(id, entries)
    # break
    if id in matched_now:
        matched_now[id].extend(entries)
    else:
        matched_now[id] = entries

# with open("matches_merged.json", "w") as f:
#     json.dump(matched_now, f, ensure_ascii=False, indent=2)

In [130]:
merged = matched_now
# rprint(dict(list(merged.items())[:2]))
rprint(len(merged))

matched_flat = []
last = first = single = None

for id, entries in merged.items():
    person_id = id
    for entry in entries:
        display_name = entry["display_name"]
        display_norm = normalise_text(display_name)

        if ", " in display_name:
            last, first = display_name.split(", ", 1)
        else:
            split = display_name.rsplit(" ", 1)
            if len(split) == 2:
                first, last = split
            else:
                single = display_name


        matched_flat.append({
            "person_id": person_id,
            **entry,
            "display_norm": display_norm,
            "last": last,
            "first": first,
            "single": single
        })

        # rprint(matched_flat)

# with open("matched_flat.json", "w") as f:
#     json.dump(matched_flat, f, ensure_ascii=False, indent=2)


1908

In [131]:
# fieldnames = list(matched_flat[0].keys())
# rprint(fieldnames)

# with open("matched_flat.csv", mode="w") as csv_file:
#     fieldnames = list(matched_flat[0].keys())
#     writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(matched_flat)